# Task 6 — Real-time webcam facial-expression demo

**5ARE0 · Assignment 2 · Group 5**  
**Notebook owner:** Henrique · **Implementation status:** scaffold, progressive local tests pending  
**Branch:** `5-task-6-real-time-webcam-demo`

## Objective

Build a testable webcam inference workflow **without choosing the team's final model prematurely**. This notebook follows the numbered, evidence-driven structure of `notebooks/task4_fuzzy_kdef.ipynb`: one checkable stage at a time, reproducible code, explicit assumptions, and a measured-results log.

**Architectural rule:** switching a `.pkl` is permitted only together with a compatible *model profile* (crop/landmarks/HOG/pixels, normalization and class mapping). `MODEL_PATH` is deliberately configurable. **The initial default is the existing fuzzy bundle** at `models/fuzzy/fuzzy_classifier.pkl`, solely for integration testing; it is **not** a recommendation that the fuzzy model be the final demo model.

The notebook can run its offline checkpoints without a `.pkl` or a webcam. Model loading, camera capture and live display require explicit opt-in. No training or retraining is performed.

**Assignment deliverables:** an executable root-level `demo.py`, `.pkl` model, notebooks, report and dependencies. A notebook by itself does not fulfill the CLI demo requirement.

**Success gates:** (1) environment, (2) artifact inventory, (3) model contract, (4) per-profile feature tests, (5) saved model load, (6) static detection and prediction, (7) camera snapshot, (8) live latency/FPS, (9) standalone CLI and report.

## 1. Discover the repository and inspect the environment

Run with the working directory at the repository root or in `notebooks/`. Only core scientific libraries and the repository's HOG extractor are imported at this stage. `dlib` and the fitted artifact will be required **only when their profile is selected**.

**Checkpoint 1:** the root should contain `feature_extraction.py`, `baseline_demo_fer2013.py`, `notebooks/` and `models/`. Record which conda environment is active.

In [1]:
from pathlib import Path
from time import perf_counter
import importlib.metadata as metadata
import platform
import sys

import cv2
import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

CWD = Path.cwd().resolve()
ROOT = next((p for p in (CWD, *CWD.parents)
             if (p / "feature_extraction.py").is_file()
             and (p / "baseline_demo_fer2013.py").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("Open this notebook inside ml-emotionalrecognition")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

def version(package):
    try:
        return metadata.version(package)
    except metadata.PackageNotFoundError:
        return "not installed"

print("Repository:", ROOT)
print("Python:", sys.version.split()[0], "| OS:", platform.platform())
print("OpenCV:", cv2.__version__, "| NumPy:", np.__version__)
print("dlib:", version("dlib"), "| PyFUME:", version("pyfume"))
print("scikit-learn:", version("scikit-learn"), "| scikit-image:", version("scikit-image"))

Repository: /Users/henriquedasilvacouto/workspace/TUe-Master_AI_and_ES/Masters-database/5ARE0_2026-GS1/Assignment_2/ml-emotionalrecognition
Python: 3.10.22 | OS: macOS-26.5.2-arm64-arm-64bit
OpenCV: 5.0.0 | NumPy: 1.24.4
dlib: 19.24.2 | PyFUME: 0.3.4
scikit-learn: 1.7.2 | scikit-image: not installed


## 2. Reuse the components that already exist

Your **local** tree (not necessarily the shared Git archive) includes:

| Component | Intended reuse | Caution |
|---|---|---|
| `models/fuzzy/fuzzy_classifier.pkl` | Integration-test model: seven PyFUME predictors + scaler | Needs original-resolution dlib facial geometry; **no 48×48 resize** |
| `models/faceAndLandmarkModels/shape_predictor_68_face_landmarks.dat` | 68-landmark predictor used by Task 4 | Use with the same geometric formulas as Task 4 |
| `models/rf_fer_model_landmarks.pkl` | Optional FER Random Forest test | Training notebook used 48×48 landmarks and a legacy `*_GTX.dat` predictor; check predictor compatibility |
| `feature_extraction.py` | FER HOG extractor | Expects raw `uint8` 48×48 input, and performs division by 255 internally |
| `baseline_demo_fer2013.py` | Reference for camera and OpenCV visual overlay | Contains outdated relative paths, hardcoded RF model and landmark logic |
| `src/preprocess.py` | Reference for KDEF crop | Not interchangeable with FER HOG or fuzzy geometry |
| `notebooks/task4_fuzzy_kdef.ipynb` | Exact fuzzy feature definitions and bundle structure | Ground truth for fuzzy `features`, `classes` and `models` |

**Checkpoint 2:** the inventory below reflects the **machine executing the notebook**. Missing files do not invalidate environment-independent tests.

In [2]:
ARTIFACTS = {
    "fuzzy bundle (available locally)": "models/fuzzy/fuzzy_classifier.pkl",
    "FER RF landmarks (available locally)": "models/rf_fer_model_landmarks.pkl",
    "FER HOG SVM (may be saved by a colleague)": "models/fer_hog_svm.pkl",
    "dlib predictor (Task 4)": "models/faceAndLandmarkModels/shape_predictor_68_face_landmarks.dat",
    "legacy dlib GTX predictor (may be unavailable)": "models/faceAndLandmarkModels/shape_predictor_68_face_landmarks_GTX.dat",
    "FER HOG extractor": "feature_extraction.py",
    "starter webcam": "baseline_demo_fer2013.py",
}
inventory = pd.DataFrame([
    {"artifact": name, "present": (ROOT / relpath).is_file(), "path": relpath}
    for name, relpath in ARTIFACTS.items()
])
print(inventory.to_string(index=False))

# OpenCV installations may not expose cv2.data.
# The fuzzy model uses dlib, so Haar is optional.

haar_dir = getattr(getattr(cv2, "data", None), "haarcascades", None)

if haar_dir is not None:
    haar_path = Path(haar_dir) / "haarcascade_frontalface_default.xml"
    print("OpenCV Haar cascade:", haar_path)
    print("Haar cascade available:", haar_path.is_file())
else:
    print("OpenCV Haar cascade: unavailable in this environment")
    print("Not required for fuzzy_kdef (using dlib)")


                                      artifact  present                                                                   path
              fuzzy bundle (available locally)     True                                      models/fuzzy/fuzzy_classifier.pkl
          FER RF landmarks (available locally)     True                                      models/rf_fer_model_landmarks.pkl
     FER HOG SVM (may be saved by a colleague)    False                                                 models/fer_hog_svm.pkl
                       dlib predictor (Task 4)     True     models/faceAndLandmarkModels/shape_predictor_68_face_landmarks.dat
legacy dlib GTX predictor (may be unavailable)    False models/faceAndLandmarkModels/shape_predictor_68_face_landmarks_GTX.dat
                             FER HOG extractor     True                                                  feature_extraction.py
                                starter webcam     True                                               baseline_

## 3. **Single configuration cell** — select model artifact and compatible profile

**Edit this cell when your colleagues select a different model.** The `MODEL_PATH` and `MODEL_PROFILE` must agree. Merely changing the path cannot adapt different input features. Do not assume the `.pkl` contains the face detector or HOG extraction code.

| `MODEL_PROFILE` | Model convention | Expected features |
|---|---|---|
| `fuzzy_kdef` | Task 4 fuzzy dictionary bundle | 7 ordered geometric features from full-resolution image |
| `fer_hog` | FER sklearn model/pipeline trained using `extract_hog` | 900 HOG values |
| `fer_landmarks` | FER sklearn model/pipeline trained on 48×48 landmarks | 136 normalized landmark coordinates |
| `fer_pixels_pca` | FER sklearn pipeline with PCA **inside the pipeline** | 2,304 scaled pixel values; PCA is applied by pipeline |

**Future Task 5 model:** update `MODEL_PATH` and choose its *actual* profile only after verifying the training notebook. If the team uses another extractor, add a new profile in §6 and test it before running the camera. **Do not identify the feature representation from the filename alone.**

Use `LOAD_MODEL=True` only when your environment has the required dependencies and your own trusted `.pkl` is present. `RUN_CAMERA_SNAPSHOT` and `RUN_LIVE_DEMO` deliberately default to `False`.

In [3]:
# ==================== EDIT CONFIGURATION HERE ====================
MODEL_PATH = ROOT / "models/fuzzy/fuzzy_classifier.pkl"
MODEL_PROFILE = "fuzzy_kdef"    # fuzzy_kdef | fer_hog | fer_landmarks | fer_pixels_pca

# Only relevant to FER-profile predictors:
MODEL_LABEL_SCHEME = "fer_numeric"  # fer_numeric | kdef_strings | fer_strings
FER_LANDMARK_VARIANT = "baseline_rf" # baseline_rf: (1,1,47,47); task1_grid: (0,0,47,47)
PREDICTOR_PATH = ROOT / "models/faceAndLandmarkModels/shape_predictor_68_face_landmarks.dat"
FER_FACE_MARGIN = 0.0         # inference adaptation; record changes if tuning

# General video controls
CAMERA_INDEX = 0
MAX_FACES = 2
MAX_LIVE_FRAMES = 300        # 'q' / ESC exits earlier
STATIC_IMAGE_PATH = None    # Optional: local consented image, Path(...)

# Enable tests intentionally; do not activate webcam from Run All.
LOAD_MODEL = True
RUN_STATIC_IMAGE_PROBE = False
RUN_CAMERA_SNAPSHOT = False
RUN_LIVE_DEMO = True
SAVE_TRIAL_LOG = False

PROFILE_FEATURE_COUNT = {
    "fuzzy_kdef": 7,
    "fer_hog": 900,
    "fer_landmarks": 136,
    "fer_pixels_pca": 2304,
}
if MODEL_PROFILE not in PROFILE_FEATURE_COUNT:
    raise ValueError(f"Unimplemented MODEL_PROFILE {MODEL_PROFILE!r}. Add a tested adapter first.")
MODEL_PATH = Path(MODEL_PATH).expanduser()
assert MAX_FACES >= 1 and FER_FACE_MARGIN >= 0
print("Model artifact:", MODEL_PATH)
print("Profile:", MODEL_PROFILE)
print("Expected feature count:", PROFILE_FEATURE_COUNT[MODEL_PROFILE])
print("Will load model:", LOAD_MODEL, "| Camera enabled:", RUN_CAMERA_SNAPSHOT or RUN_LIVE_DEMO)

Model artifact: /Users/henriquedasilvacouto/workspace/TUe-Master_AI_and_ES/Masters-database/5ARE0_2026-GS1/Assignment_2/ml-emotionalrecognition/models/fuzzy/fuzzy_classifier.pkl
Profile: fuzzy_kdef
Expected feature count: 7
Will load model: True | Camera enabled: True


## 4. Shared FER crop, sizing and label conventions

The FER2013 dataset provides 48×48 grayscale face patches. FER HOG, FER landmarks and FER PCA must first transform the webcam's face ROI to **48×48 `uint8`**. This preprocessing does **not** apply to the fuzzy model, whose training pipeline detected landmarks on original-resolution KDEF images.

Haar-detected bounding boxes need to stay within frame bounds. A margin is an experimental *inference change*, not proof of identical FER preprocessing.

**Checkpoint 4:** test clipping and resizing without a real camera.

In [4]:
FER_LABELS = ("Angry", "Disgust", "Fear", "Happy", "Sad", "Surprise", "Neutral")
KDEF_LABEL_TO_FER = {
    "afraid": "Fear", "fear": "Fear", "angry": "Angry",
    "disgust": "Disgust", "happy": "Happy", "sad": "Sad",
    "surprise": "Surprise", "neutral": "Neutral",
}


def expand_and_clip_box(box, image_shape, margin=0.0):
    x, y, w, h = map(int, box)
    image_height, image_width = image_shape[:2]
    if min(w, h) <= 0 or margin < 0:
        raise ValueError(f"Invalid rectangle / margin: {box}, {margin}")
    dx, dy = round(w * margin), round(h * margin)
    x1, y1 = max(0, x - dx), max(0, y - dy)
    x2 = min(image_width, x + w + dx)
    y2 = min(image_height, y + h + dy)
    if x1 >= x2 or y1 >= y2:
        raise ValueError("Empty face ROI")
    return (x1, y1, x2, y2)


def prepare_face_fer(gray, box, margin=FER_FACE_MARGIN):
    """OpenCV bounding box -> FER-compatible 48x48 uint8 face patch."""
    if gray.ndim != 2 or gray.dtype != np.uint8:
        raise TypeError("Expect grayscale uint8 frame")
    x1, y1, x2, y2 = expand_and_clip_box(box, gray.shape, margin)
    crop = gray[y1:y2, x1:x2]
    method = cv2.INTER_AREA if min(crop.shape) >= 48 else cv2.INTER_LINEAR
    face48 = cv2.resize(crop, (48, 48), interpolation=method)
    return face48, (x1, y1, x2, y2)

synthetic_gray = np.arange(120 * 160, dtype=np.uint8).reshape(120, 160)
synthetic_face, synthetic_bounds = prepare_face_fer(synthetic_gray, (20, 25, 65, 70))
assert synthetic_face.shape == (48, 48) and synthetic_face.dtype == np.uint8
assert synthetic_bounds == (20, 25, 85, 95)
assert expand_and_clip_box((0, 0, 80, 80), synthetic_gray.shape, 0.2) == (0, 0, 96, 96)
print("PASS: crop bounds and FER sizing")

PASS: crop bounds and FER sizing


## 5. Exact fuzzy geometric feature extraction (reused from Task 4)

`task4_fuzzy_kdef.ipynb` uses the **68 dlib landmarks from original-resolution grayscale images**. These seven feature formulas and their ordering must be preserved. Specifically, `smile_ratio` is mouth-width divided by mouth-height; it is **not** a calibrated smile confidence.

Fuzzy classes are `afraid, angry, disgust, happy, neutral, sad, surprise` and the saved bundle includes a training-fitted `MinMaxScaler` plus seven binary PyFUME models. Their numeric outputs are **not calibrated probabilities**.

**Checkpoint 5:** geometric measurements have the correct names and produce finite values for realistic landmark coordinates. We do **not** synthesize claims about model accuracy.

In [5]:
FUZZY_FEATURE_COLS = [
    "mouth_width", "mouth_opening", "left_eye_opening", "right_eye_opening",
    "eyebrow_height", "smile_ratio", "mouth_corner_asymmetry",
]
FUZZY_CLASSES = ["afraid", "angry", "disgust", "happy", "neutral", "sad", "surprise"]


def geometric_features(points):
    """Exact seven normalized ratios from Task 4; input (68, 2) x/y points."""
    points = np.asarray(points, dtype=float)
    if points.shape != (68, 2):
        raise ValueError(f"Expected (68,2) dlib coordinates; received {points.shape}")
    distance = lambda a, b: float(np.linalg.norm(points[a] - points[b]))
    eps = 1e-6
    face_width = max(distance(0, 16), eps)
    mouth_width = distance(48, 54)
    mouth_height = distance(51, 57)
    values = {
        "mouth_width": mouth_width / face_width,
        "mouth_opening": distance(62, 66) / face_width,
        "left_eye_opening": (distance(37, 41) + distance(38, 40))
                            / (2 * max(distance(36, 39), eps)),
        "right_eye_opening": (distance(43, 47) + distance(44, 46))
                             / (2 * max(distance(42, 45), eps)),
        "eyebrow_height": (distance(19, 37) + distance(24, 44)) / (2 * face_width),
        "smile_ratio": mouth_width / max(mouth_height, eps),
        "mouth_corner_asymmetry": abs(float(points[48, 1] - points[54, 1])) / face_width,
    }
    if not np.isfinite(list(values.values())).all():
        raise ValueError("Non-finite geometric ratios")
    return values

assert list(geometric_features(np.column_stack([np.arange(68),
                                                 np.arange(68) + 10])).keys()) == FUZZY_FEATURE_COLS
print("PASS: fuzzy feature names/order and finite-value check")

PASS: fuzzy feature names/order and finite-value check


## 6. FER feature adapters: HOG, PCA-ready pixels and 68-point landmarks

These are supported as **alternative inference profiles**; they are not fed to the fuzzy bundle.

- **HOG:** `feature_extraction.extract_hog()` expects 48×48 `uint8` and divides by 255 internally. Expected output: 900 features.
- **PCA:** `01_fer_baseline_grid.ipynb` starts with `image.reshape(-1) / 255` (2,304 floats); its fitted sklearn pipeline performs PCA. Never refit PCA on webcam data.
- **FER landmarks:** 136 normalized coordinates on the full 48×48 face patch. The existing starter RF model originates from a legacy baseline using rectangle `(1, 1, 47, 47)` and a `*_GTX.dat` predictor, whereas Task 1's grid implementation uses `(0, 0, 47, 47)`. **Match the variant and predictor actually used during training.** Presence of the non-GTX `.dat` alone does not prove interchangeability with the GTX predictor.

**Checkpoint 6:** HOG and pixels work without a dlib model. Landmark extraction becomes testable once its predictor and variant are confirmed.

In [6]:
def fer_hog_features(face48):
    from feature_extraction import extract_hog

    if face48.shape != (48, 48) or face48.dtype != np.uint8:
        raise TypeError("FER HOG expects uint8 48x48")

    features = extract_hog(face48[np.newaxis, ...])

    if features.shape != (1, 900) or not np.isfinite(features).all():
        raise ValueError(f"Unexpected HOG shape: {features.shape}")

    return features


def fer_pixel_features(face48):
    if face48.shape != (48, 48) or face48.dtype != np.uint8:
        raise TypeError("FER PCA input expects uint8 48x48")
    return face48.reshape(1, 2304).astype(np.float32) / 255.0


def fer_landmark_features(face48, predictor, variant=FER_LANDMARK_VARIANT):
    """Match the FER training notebook's dlib rectangle and normalization."""
    import dlib
    if variant == "baseline_rf":
        rect = dlib.rectangle(left=1, top=1, right=47, bottom=47)
    elif variant == "task1_grid":
        rect = dlib.rectangle(left=0, top=0, right=47, bottom=47)
    else:
        raise ValueError(f"Unsupported landmark training variant: {variant}")
    shape = predictor(face48, rect)
    pts = np.asarray([(p.x, p.y) for p in shape.parts()], dtype=np.float32)
    if pts.shape != (68, 2):
        raise ValueError(f"Predictor returned {pts.shape} points")
    width, height = rect.right() - rect.left(), rect.bottom() - rect.top()
    normalized = np.column_stack([
        (pts[:, 0] - 0.5 * width) / width,
        (pts[:, 1] - 0.5 * height) / height,
    ]).astype(np.float32)
    normalized -= normalized.mean(axis=0, keepdims=True)
    scale = np.max(np.abs(normalized), axis=0, keepdims=True)
    if np.any(scale <= 0):
        raise ValueError("Degenerate landmark configuration")
    normalized /= scale
    return normalized.reshape(1, 136)

# ---------------------------------------------------------
# Validate FER pixel features
# ---------------------------------------------------------

pixels_unit = fer_pixel_features(synthetic_face)

assert pixels_unit.shape == (1, 2304)
assert 0 <= pixels_unit.min() <= pixels_unit.max() <= 1

print("PASS: FER pixel feature extraction")

# ---------------------------------------------------------
# Validate HOG only when selected
# ---------------------------------------------------------

if MODEL_PROFILE == "fer_hog":
    hog_unit = fer_hog_features(synthetic_face)

    assert hog_unit.shape == (1, 900)
    assert hog_unit.dtype == np.float32

    print("PASS: FER HOG feature extraction")
else:
    print("SKIPPED: HOG validation (not selected model)")

# ---------------------------------------------------------
# Validate landmark feature function
# ---------------------------------------------------------

print("PASS: FER feature adapter functions defined")

PASS: FER pixel feature extraction
SKIPPED: HOG validation (not selected model)
PASS: FER feature adapter functions defined


## 7. Load and validate the selected `.pkl` (optional, explicit opt-in)

**This is the only model-loading stage.** Only load artifacts created by trusted team members. A malicious `.pkl` can execute arbitrary code when deserialized.

- `fuzzy_kdef`: expects a dictionary with `models`, `scaler`, `features`, `classes`; validate the feature order and all seven binary estimators.
- FER profiles: expect a fitted sklearn estimator / pipeline with `.predict()` and class labels; check `n_features_in_` if exposed.
- If Task 5 uses different feature extraction, **stop and add a new adapter** rather than guessing.

**Important:** PyFUME bundles may require `emotion_fuzzy` (Python 3.10 / NumPy 1.24.4 / `pyfume==0.3.4`). The general environment is not guaranteed to deserialize them correctly. Use the compatible environment for the fuzzy trial.

**Checkpoint 7:** successful deserialization, validated schema / input width; no claims about real webcam accuracy yet.

In [7]:
def check_loaded_artifact(artifact, profile, label_scheme=MODEL_LABEL_SCHEME):
    """Fail early on incompatible artifact types or obvious width/label mismatches."""
    if profile == "fuzzy_kdef":
        required = {"models", "scaler", "features", "classes"}
        if not isinstance(artifact, dict) or not required.issubset(artifact):
            raise TypeError("Fuzzy model requires Task 4 dictionary bundle")
        if list(artifact["features"]) != FUZZY_FEATURE_COLS:
            raise ValueError("Fuzzy feature order disagrees with Task 4")
        if list(artifact["classes"]) != FUZZY_CLASSES:
            raise ValueError("Unexpected fuzzy class ordering")
        if set(artifact["models"]) != set(FUZZY_CLASSES):
            raise ValueError("Missing or additional fuzzy one-vs-rest models")
        if not callable(getattr(artifact["scaler"], "transform", None)):
            raise TypeError("Fuzzy scaler lacks transform()")
        if any(not callable(getattr(artifact["models"][cls], "predict_label", None))
               for cls in FUZZY_CLASSES):
            raise TypeError("Fuzzy binary model lacks predict_label()")
        scaler_width = getattr(artifact["scaler"], "n_features_in_", None)
        if scaler_width is not None and scaler_width != 7:
            raise ValueError(f"Expected scaler with 7 columns, got {scaler_width}")
        return "Fuzzy Task 4 bundle: 7 features -> 7 binary PyFUME outputs"

    if not callable(getattr(artifact, "predict", None)):
        raise TypeError("FER profile requires a fitted estimator with predict()")
    expected_width = PROFILE_FEATURE_COUNT[profile]
    model_width = getattr(artifact, "n_features_in_", None)
    if model_width is not None and model_width != expected_width:
        raise ValueError(f"{profile} produces {expected_width} features, model expects {model_width}")
    classes = getattr(artifact, "classes_", None)
    if classes is None:
        raise ValueError("Model classes_ missing; verify class mapping before inference")
    if label_scheme == "fer_numeric":
        if not all(isinstance(v, (int, np.integer)) and 0 <= int(v) < 7 for v in classes):
            raise ValueError("FER numeric mapping requires integer class labels 0..6")
    elif label_scheme == "fer_strings":
        if not set(map(str, classes)).issubset(set(FER_LABELS)):
            raise ValueError("Unexpected FER string labels")
    elif label_scheme == "kdef_strings":
        if not set(map(str, classes)).issubset(set(KDEF_LABEL_TO_FER)):
            raise ValueError("Unexpected KDEF string labels")
    else:
        raise ValueError(f"Unknown MODEL_LABEL_SCHEME: {label_scheme}")
    return f"Sklearn model: {profile}, {expected_width} features, classes={list(classes)}"

selected_model = None
if LOAD_MODEL:
    if not MODEL_PATH.is_file():
        raise FileNotFoundError(f"Missing selected model: {MODEL_PATH}")
    selected_model = joblib.load(MODEL_PATH)  # trusted local team artifact ONLY
    print("PASS:", check_loaded_artifact(selected_model, MODEL_PROFILE))
else:
    print("SKIPPED model load: set LOAD_MODEL=True only when ready for a local trial")

PASS: Fuzzy Task 4 bundle: 7 features -> 7 binary PyFUME outputs


## 8. Initialize face detection and required landmark predictors

Detection strategy is **profile-dependent**:

- **Fuzzy:** `dlib.get_frontal_face_detector()` on the *original-resolution* grayscale frame with one upsample, matching Task 4. The dlib `rectangle` goes directly into its 68-point predictor; **no 48×48 resize or Haar crop**.
- **FER:** OpenCV Haar cascade, matching the starter demo approach, followed by the 48×48 patch. A separate predictor is needed **only** for the `fer_landmarks` profile.

The OpenCV package contains a Haar XML at `cv2.data.haarcascades`, so an additional XML file is not required in your tree.

**Checkpoint 8:** detector initialization; a black synthetic frame returns zero faces. Landmark predictor is loaded lazily when the selected profile needs it.

In [8]:
def make_detector(profile):
    if profile == "fuzzy_kdef":
        import dlib
        return dlib.get_frontal_face_detector()
    xml = Path(cv2.data.haarcascades) / "haarcascade_frontalface_default.xml"
    if not xml.is_file():
        raise FileNotFoundError(f"OpenCV cascade missing: {xml}")
    detector = cv2.CascadeClassifier(str(xml))
    if detector.empty():
        raise RuntimeError(f"Unable to read Haar cascade: {xml}")
    return detector


def initialize_predictor(profile):
    if profile not in ("fuzzy_kdef", "fer_landmarks"):
        return None
    import dlib
    if not PREDICTOR_PATH.is_file():
        raise FileNotFoundError(f"Missing dlib predictor: {PREDICTOR_PATH}")
    return dlib.shape_predictor(str(PREDICTOR_PATH))


def detect_faces(gray, detector, profile):
    if gray.ndim != 2 or gray.dtype != np.uint8:
        raise TypeError("Expected uint8 grayscale image")
    if profile == "fuzzy_kdef":
        rectangles = list(detector(gray, 1))
        return sorted(rectangles, key=lambda r: r.width() * r.height(), reverse=True)
    rectangles = detector.detectMultiScale(gray, scaleFactor=1.3,
                                           minNeighbors=5, minSize=(50, 50))
    return sorted([tuple(map(int, r)) for r in rectangles],
                  key=lambda r: r[2] * r[3], reverse=True)


def box_for_display(rect, image_shape, profile):
    if profile == "fuzzy_kdef":
        x, y, w, h = rect.left(), rect.top(), rect.width(), rect.height()
        return expand_and_clip_box((x, y, w, h), image_shape)
    return expand_and_clip_box(rect, image_shape)

detector = make_detector(MODEL_PROFILE)
predictor = initialize_predictor(MODEL_PROFILE) if PREDICTOR_PATH.is_file() else None
# When no predictor is present in an archived ZIP, offline detector/unit tests still work.
if MODEL_PROFILE == "fuzzy_kdef" and predictor is None:
    print("Predictor not in current snapshot: geometry requires PREDICTOR_PATH locally")
print("PASS: detector initialized:", type(detector).__name__)
print("Synthetic empty-frame detections:", len(detect_faces(np.zeros((120, 120), dtype=np.uint8), detector, MODEL_PROFILE)))

PASS: detector initialized: fhog_object_detector
Synthetic empty-frame detections: 0


## 9. One prediction adapter shared by still-image and webcam tests

The selected `.pkl` is never called with the wrong feature family. For the fuzzy bundle, transform the **ordered seven-column DataFrame** using its saved scaler, then evaluate each one-versus-rest fuzzy model using `predict_label`. Pick the highest numeric output. **Do not display the output as probability or confidence.**

For FER models, crop to 48×48, extract the configured features, and apply the fitted sklearn estimator. Return the decoded class and processing latency. A face-detection failure is distinct from a classification failure.

**Checkpoint 9:** verify dispatch and class decoding; actual predictions require `LOAD_MODEL=True`.

In [9]:
def decode_fer_label(raw, label_scheme):
    if label_scheme == "fer_numeric":
        if not isinstance(raw, (int, np.integer)) or not 0 <= int(raw) < 7:
            raise ValueError(f"Unexpected FER numeric prediction {raw!r}")
        return FER_LABELS[int(raw)]
    if label_scheme == "fer_strings":
        if str(raw) not in FER_LABELS:
            raise ValueError(f"Unknown FER class {raw!r}")
        return str(raw)
    if label_scheme == "kdef_strings":
        if str(raw) not in KDEF_LABEL_TO_FER:
            raise ValueError(f"Unknown KDEF class {raw!r}")
        return KDEF_LABEL_TO_FER[str(raw)]
    raise ValueError(f"Unknown label scheme {label_scheme}")


def infer_one_face(gray, face_rect, artifact, predictor_model, profile=MODEL_PROFILE):
    """Compute features and predict for one already-detected face."""
    if artifact is None:
        raise RuntimeError("Set LOAD_MODEL=True before requesting inference")
    started = perf_counter()
    if profile == "fuzzy_kdef":
        if predictor_model is None:
            raise RuntimeError("Fuzzy predictor not initialized")
        shape = predictor_model(gray, face_rect)
        landmarks = np.asarray([(p.x, p.y) for p in shape.parts()], dtype=float)
        geometric = geometric_features(landmarks)
        feature_frame = pd.DataFrame([[geometric[key] for key in artifact["features"]]],
                                     columns=artifact["features"])
        scaled = artifact["scaler"].transform(feature_frame)
        scores = np.asarray([
            float(np.asarray(artifact["models"][cls].predict_label(scaled), dtype=float).ravel()[0])
            for cls in artifact["classes"]
        ], dtype=float)
        if not np.isfinite(scores).all():
            raise ValueError("Fuzzy model returned non-finite outputs")
        raw_label = artifact["classes"][int(np.argmax(scores))]
        return {
            "label": KDEF_LABEL_TO_FER[raw_label], "raw_class": raw_label,
            "features_shape": scaled.shape,
            "inference_ms": (perf_counter() - started) * 1000,
            "details": "Fuzzy raw scores are NOT calibrated probabilities",
        }

    face48, bounds = prepare_face_fer(gray, face_rect, FER_FACE_MARGIN)
    if profile == "fer_hog":
        X = fer_hog_features(face48)
    elif profile == "fer_pixels_pca":
        X = fer_pixel_features(face48)
    elif profile == "fer_landmarks":
        if predictor_model is None:
            raise RuntimeError("Landmark profile needs a matching dlib predictor")
        X = fer_landmark_features(face48, predictor_model)
    else:
        raise ValueError(f"Unsupported profile: {profile}")
    raw_values = np.asarray(artifact.predict(X)).ravel()
    if raw_values.shape != (1,):
        raise ValueError(f"Expected 1 prediction, got {raw_values.shape}")
    raw_label = raw_values[0]
    return {
        "label": decode_fer_label(raw_label, MODEL_LABEL_SCHEME),
        "raw_class": raw_label.item() if hasattr(raw_label, "item") else raw_label,
        "features_shape": X.shape,
        "face48": face48,
        "bounds": bounds,
        "inference_ms": (perf_counter() - started) * 1000,
        "details": "Sklearn class prediction (not a probability)",
    }

assert decode_fer_label(2, "fer_numeric") == "Fear"
assert decode_fer_label("afraid", "kdef_strings") == "Fear"
assert decode_fer_label("Happy", "fer_strings") == "Happy"
print("PASS: label mapping and model-specific inference dispatch defined")

PASS: label mapping and model-specific inference dispatch defined


## 10. Static-image checkpoint — **before** using the laptop webcam

Select one locally available image with permission to process it. With a loaded model the notebook will run all steps. If the model is not yet loaded, detection and crop visualization still work; this is useful for isolating camera/detection issues.

For fuzzy, inspect original-resolution face detection and draw the 68 landmarks **before claiming feature parity**. For FER, visually inspect the actual 48×48 patch that will be classified.

**Checkpoint 10:** record the number of detected faces, size, crop/landmark quality, and any exceptions. Static test is optional to permit headless execution.

In [10]:
def inspect_bgr_image(image_bgr, artifact=None, show=True):
    if image_bgr is None or image_bgr.ndim != 3 or image_bgr.shape[-1] != 3:
        raise ValueError("Expected HxWx3 BGR image")
    gray = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2GRAY)
    found = detect_faces(gray, detector, MODEL_PROFILE)
    overlay = image_bgr.copy()
    results = []
    for rect in found[:MAX_FACES]:
        x1, y1, x2, y2 = box_for_display(rect, gray.shape, MODEL_PROFILE)
        record = {"bounds": (x1, y1, x2, y2), "prediction": None}
        if MODEL_PROFILE == "fuzzy_kdef" and predictor is not None:
            # Original-resolution landmark overlay: visual parity check with Task 4.
            shape = predictor(gray, rect)
            points = np.asarray([(pt.x, pt.y) for pt in shape.parts()], dtype=int)
            record["landmarks"] = points
            for point_x, point_y in points:
                cv2.circle(overlay, (int(point_x), int(point_y)), 2, (0, 220, 0), -1)
        elif MODEL_PROFILE != "fuzzy_kdef":
            record["face48"], _ = prepare_face_fer(gray, rect, FER_FACE_MARGIN)
        if artifact is not None:
            record["prediction"] = infer_one_face(gray, rect, artifact, predictor)
        cv2.rectangle(overlay, (x1, y1), (x2, y2), (0, 200, 230), 2)
        label = record["prediction"]["label"] if record["prediction"] else "Face detected"
        cv2.putText(overlay, label, (x1, max(22, y1 - 8)),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.65, (0, 220, 220), 2)
        results.append(record)
    if show:
        plt.figure(figsize=(9, 5))
        plt.imshow(cv2.cvtColor(overlay, cv2.COLOR_BGR2RGB))
        plt.axis("off")
        plt.title(f"Detected {len(found)} face(s) | profile: {MODEL_PROFILE}")
        plt.show()
        if results and MODEL_PROFILE != "fuzzy_kdef":
            plt.figure(figsize=(3, 3))
            plt.imshow(results[0]["face48"], cmap="gray", vmin=0, vmax=255)
            plt.title("FER 48x48 input")
            plt.axis("off")
            plt.show()
    return results

if RUN_STATIC_IMAGE_PROBE:
    if STATIC_IMAGE_PATH is None:
        raise ValueError("Set STATIC_IMAGE_PATH in §3 first")
    bgr = cv2.imread(str(Path(STATIC_IMAGE_PATH).expanduser()))
    if bgr is None:
        raise FileNotFoundError(STATIC_IMAGE_PATH)
    static_results = inspect_bgr_image(bgr, artifact=selected_model)
    print("Static detected:", len(static_results))
else:
    print("SKIPPED static-image probe. Set STATIC_IMAGE_PATH and RUN_STATIC_IMAGE_PROBE=True.")

SKIPPED static-image probe. Set STATIC_IMAGE_PATH and RUN_STATIC_IMAGE_PROBE=True.


## 11. Webcam access — capture exactly one frame (opt-in)

OpenCV captures from the laptop **only when `RUN_CAMERA_SNAPSHOT=True`**. The camera should be in the same local session as your Jupyter kernel (not a remote server). On macOS, grant the application running Python permission to use the camera.

This checkpoint tests the camera independently of the selected model. Display the snapshot through `matplotlib`, not continuous `cv2.imshow`.

**Checkpoint 11:** camera opens; frame is valid; detection returns expected face boxes. Raw snapshots are not saved to disk by default.

In [11]:
def open_webcam(index=CAMERA_INDEX):
    backends = ([cv2.CAP_AVFOUNDATION, cv2.CAP_ANY]
                if sys.platform == "darwin" else [cv2.CAP_ANY])
    for backend in backends:
        camera = cv2.VideoCapture(index, backend)
        if camera.isOpened():
            return camera
        camera.release()
    raise RuntimeError(f"Camera {index} unavailable: check OS permissions, index and local Jupyter kernel")


def capture_one_frame(index=CAMERA_INDEX):
    camera = open_webcam(index)
    try:
        ok, frame = camera.read()
        if not ok or frame is None:
            raise RuntimeError("Camera opened but did not return a frame")
        return frame
    finally:
        camera.release()

if RUN_CAMERA_SNAPSHOT:
    camera_snapshot = capture_one_frame()
    print("Captured BGR frame:", camera_snapshot.shape)
    camera_preview = inspect_bgr_image(camera_snapshot, artifact=selected_model)
    print("Faces processed:", len(camera_preview))
else:
    print("SKIPPED camera capture. Set RUN_CAMERA_SNAPSHOT=True on your laptop.")

SKIPPED camera capture. Set RUN_CAMERA_SNAPSHOT=True on your laptop.


## 12. Live inference loop — opt-in, finite duration, measured performance

This stage requires a compatible loaded model and successful one-frame capture. It displays the image with detected face boxes and **predicted facial-expression labels**. Do **not** treat these labels as a medical or psychological measurement of emotion.

Recorded metrics: number of frames, fraction containing detected faces, classified faces, elapsed time, processed FPS, detection latency, and feature+predict latency (mean/p95). FPS and latency are measured only during the actual local test; values remain blank otherwise.

To stop press **`q` or Escape** in the OpenCV window. The default test terminates after 300 frames; resources are released even on errors. OpenCV window support varies by macOS/VS Code/Jupyter environment. Prefer running the final `demo.py` in a normal terminal if notebook windows are unreliable.

**Checkpoint 12:** record stability, light/pose/occlusion failures, and whether FPS is adequate on the actual laptop.

In [18]:
def run_live_webcam(artifact, index=CAMERA_INDEX, max_frames=MAX_LIVE_FRAMES):
    if artifact is None:
        raise RuntimeError("Load the selected trained model first (LOAD_MODEL=True)")
    if MODEL_PROFILE in ("fuzzy_kdef", "fer_landmarks") and predictor is None:
        raise RuntimeError("Matching dlib predictor unavailable")
    camera = open_webcam(index)
    counts = {"frames": 0, "frames_with_faces": 0, "classified_faces": 0, "face_failures": 0}
    detection_ms = []
    inference_ms = []
    initial = perf_counter()

    # Create and raise the webcam window
    WINDOW_NAME = "Task 6 - Facial expression demo (q/ESC to stop)"

    cv2.namedWindow(WINDOW_NAME, cv2.WINDOW_NORMAL)

    try:
        cv2.setWindowProperty(
            WINDOW_NAME,
            cv2.WND_PROP_TOPMOST,
            1
        )
    except (cv2.error, AttributeError):
        # Some macOS OpenCV backends do not support this property
        pass
    try:
        while max_frames is None or counts["frames"] < max_frames:
            ok, frame = camera.read()
            if not ok or frame is None:
                print("Camera stopped delivering frames")
                break
            counts["frames"] += 1
            gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
            detect_start = perf_counter()
            faces = detect_faces(gray, detector, MODEL_PROFILE)
            detection_ms.append((perf_counter() - detect_start) * 1000)
            if faces:
                counts["frames_with_faces"] += 1
            for rect in faces[:MAX_FACES]:
                try:
                    result = infer_one_face(gray, rect, artifact, predictor)
                    x1, y1, x2, y2 = box_for_display(rect, gray.shape, MODEL_PROFILE)
                    counts["classified_faces"] += 1
                    inference_ms.append(result["inference_ms"])
                    cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 220, 220), 2)
                    cv2.putText(frame, result["label"], (x1, max(20, y1 - 8)),
                                cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 220, 220), 2)
                except (ValueError, TypeError, RuntimeError, cv2.error) as exc:
                    counts["face_failures"] += 1
                    print("Face skipped:", exc)
            elapsed = max(perf_counter() - initial, 1e-9)
            fps = counts["frames"] / elapsed
            cv2.putText(frame, f"Processed FPS: {fps:.1f}", (10, 28),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.72, (0, 200, 0), 2)
            cv2.imshow(WINDOW_NAME, frame)
            if (cv2.waitKey(1) & 0xff) in (ord("q"), 27):
                break
    finally:
        # Always release the camera
        camera.release()

        # Request closure of all OpenCV windows
        cv2.destroyAllWindows()

        # Process pending GUI events on macOS
        for _ in range(20):
            cv2.waitKey(1)

        print("Camera released and OpenCV cleanup completed.")
    elapsed = max(perf_counter() - initial, 1e-9)
    mean_or_none = lambda values: round(float(np.mean(values)), 2) if values else None
    p95_or_none = lambda values: round(float(np.percentile(values, 95)), 2) if values else None
    return {
        "model_path": str(MODEL_PATH), "model_profile": MODEL_PROFILE,
        "frames": counts["frames"],
        "frames_with_faces": counts["frames_with_faces"],
        "faces_classified": counts["classified_faces"],
        "face_processing_failures": counts["face_failures"],
        "elapsed_seconds": round(elapsed, 2),
        "processed_fps": round(counts["frames"] / elapsed, 2),
        "mean_detector_ms_per_frame": mean_or_none(detection_ms),
        "mean_feature_predict_ms_per_face": mean_or_none(inference_ms),
        "p95_feature_predict_ms_per_face": p95_or_none(inference_ms),
        "camera_index": index, "platform": platform.platform(),
        "fer_face_margin": FER_FACE_MARGIN if MODEL_PROFILE != "fuzzy_kdef" else None,
    }

live_metrics = None
if RUN_LIVE_DEMO:
    live_metrics = run_live_webcam(selected_model)
    print(pd.Series(live_metrics).to_string())
else:
    print("SKIPPED live test. Enable after validating your model and a camera snapshot.")

Camera released and OpenCV cleanup completed.
model_path                          /Users/henriquedasilvacouto/workspace/TUe-Mast...
model_profile                                                              fuzzy_kdef
frames                                                                              8
frames_with_faces                                                                   8
faces_classified                                                                    8
face_processing_failures                                                            0
elapsed_seconds                                                                  6.22
processed_fps                                                                    1.29
mean_detector_ms_per_frame                                                     701.88
mean_feature_predict_ms_per_face                                                14.23
p95_feature_predict_ms_per_face                                                 18.35
camera_i

# —->> Implementation Status & What's Missing to Conclude Task 6


HC to continue this once the team decide which model to pick!!


**Status: WORKING PROTOTYPE — Final model integration pending**

The initial real-time webcam demonstration has been successfully implemented and tested using the Task 4 fuzzy classifier (`models/fuzzy/fuzzy_classifier.pkl`).

The notebook was intentionally designed to separate webcam processing from model-specific inference, allowing the final classifier to be integrated once the team completes model selection.

#### Completed

- [x] Configurable model path and inference profile.
- [x] Model loading and compatibility validation.
- [x] Webcam initialization and 1920 × 1080 frame capture.
- [x] Real-time face detection using dlib.
- [x] Facial landmark extraction and geometric feature computation.
- [x] Successful fuzzy inference on webcam images.
- [x] Correctly identified happy and sad expressions in initial manual tests.
- [x] Continuous webcam processing with bounding boxes and predicted labels.
- [x] Basic FPS, detection latency and inference latency measurements.
- [x] Camera release and OpenCV window cleanup on macOS.

#### Preliminary Performance

Initial live test with the fuzzy classifier:

| Metric | Result |
|---|---:|
| Frames processed | 15 |
| Successfully classified frames | 15 |
| Face-processing failures | 0 |
| Average processing FPS | 1.17 |
| Mean face-detection latency | 693.03 ms |
| Mean feature extraction + prediction | 17.00 ms |

**Observation:** Face detection is currently the primary performance bottleneck. Classification is considerably faster than detection. These results are preliminary and do not represent a complete benchmark or measured classification accuracy.

#### Pending — Required to Complete Task 6

- [ ] **Final model selection:** Confirm the best-performing model from the team's experiments (particularly Task 5).
- [ ] **Model integration:** Update `MODEL_PATH`, `MODEL_PROFILE` and the inference adapter for the selected `.pkl`.
- [ ] **Pipeline compatibility:** Verify that face cropping, feature extraction, normalization and class mapping exactly reproduce the final model's training pipeline.
- [ ] **Dependency cleanup:** Remove unused fuzzy, HOG or landmark dependencies once the final configuration is established.
- [ ] **Performance optimization:** Reduce face-detection overhead, potentially by detecting on downscaled frames and mapping coordinates back to the original image.
- [ ] **Extended validation:** Test multiple emotions, lighting conditions, head positions, distances and longer continuous sessions.
- [ ] **Performance benchmarking:** Record final FPS, latency, detection failures and observed classification behavior using the selected model.
- [ ] **Standalone application:** Create and validate `demo.py`, executable from the command line without Jupyter.
- [ ] **Reproducibility:** Finalize environment requirements, model paths, startup instructions and shutdown behavior.
- [ ] **Report contribution:** Document webcam performance, practical limitations, adaptations and observed failure cases for the Task 6 report section.

#### Required Input From the Team

Before final integration, the Task 6 implementation needs:

1. The selected trained `.pkl` artifact.
2. The corresponding training and feature-extraction implementation.
3. The exact preprocessing, normalization and emotion-label mapping.
4. Any model-specific Python dependencies and environment requirements.

#### Implementation Strategy

The current fuzzy implementation serves as a validated proof of concept for the complete camera-to-classification workflow.

**The goal is to preserve the camera capture, visualization, benchmarking and testing infrastructure while replacing only the components required by the final model.**

The notebook should remain the documented experimental environment. The final deliverable will be a simplified, standalone `demo.py` based on the validated pipeline.

**Task 6 is functionally demonstrated but not yet ready for final submission.**

---

---


---

---

## PLACEHOLDER 13. Manual trial log and observations ---->>>> PENDING FOR NEXT REFACTOR (after we select which model to use)

Run repeatable manual trials for **neutral**, **smile**, **surprise**, **raised eyebrows**, **different light**, **head rotation**, **glasses/occlusion** and **no face in view** (as appropriate). Record *observed facial expressions*, not unverifiable inner emotions. Avoid displaying fuzzy scores as confidence.

Do not use webcam trials as a substitute for held-out dataset evaluation; there are no reliable ground-truth labels in an uncontrolled live demo. Include failure examples and practical engineering adjustments in the report.

The default log stores **no photographs or raw camera frames**. Save it only after adding your own measured observations.

In [19]:
trial_columns = [
    "trial_id", "model_profile", "condition", "observed_expression",
    "predicted_expression", "face_detected", "qualitative_match",
    "lighting", "pose_or_occlusion", "processed_fps", "inference_ms", "notes"
]
trial_log = pd.DataFrame(columns=trial_columns)
# TODO, after local observation:
# trial_log.loc[len(trial_log)] = ["T01", MODEL_PROFILE, "neutral frontal", "neutral",
#     "[ACTUAL PREDICTION]", True, "[OBSERVED]", "normal", "frontal", None, None, "..."]

print("Measured metrics:", live_metrics if live_metrics else "[NOT YET MEASURED]")
print("Observations:", len(trial_log), "rows")
if SAVE_TRIAL_LOG and not trial_log.empty:
    out = ROOT / "outputs/webcam/task6_trial_log.csv"
    out.parent.mkdir(parents=True, exist_ok=True)
    trial_log.to_csv(out, index=False)
    print("Saved:", out)

Measured metrics: {'model_path': '/Users/henriquedasilvacouto/workspace/TUe-Master_AI_and_ES/Masters-database/5ARE0_2026-GS1/Assignment_2/ml-emotionalrecognition/models/fuzzy/fuzzy_classifier.pkl', 'model_profile': 'fuzzy_kdef', 'frames': 8, 'frames_with_faces': 8, 'faces_classified': 8, 'face_processing_failures': 0, 'elapsed_seconds': 6.22, 'processed_fps': 1.29, 'mean_detector_ms_per_frame': 701.88, 'mean_feature_predict_ms_per_face': 14.23, 'p95_feature_predict_ms_per_face': 18.35, 'camera_index': 0, 'platform': 'macOS-26.5.2-arm64-arm-64bit', 'fer_face_margin': None}
Observations: 0 rows


## PLACEHOLDER 14. Finalize the standalone `demo.py` **after** the selected-model adapter is tested

**Required by the official assignment:** a root-level CLI Python program opening the laptop camera with the selected model. Do not submit only this notebook. To avoid untested duplication, the CLI implementation is a **TODO until the final `.pkl` is agreed and §7–12 pass**.

Suggested command shape (not yet implemented):

```bash
python demo.py --model models/<your-selected-model>.pkl --profile fuzzy_kdef --camera 0
```

The demo should expose `--model`, `--profile` and `--camera`; validate the artifact before activating the camera; and display clear instructions for quitting. The `.pkl` and predictor must be included in delivery (or accompanied by exact reproducible setup instructions, consistent with size constraints). Do not import the notebook at runtime.

**Final acceptance:** fresh environment + supplied files + a single reproducible command + observed webcam operation + clear error messages when camera/model dependencies are missing.

In [ ]:
# TODO (not executed): implement ROOT / "demo.py" after the model selection.
# Implementation should reuse the validated functions from §§4–12.
# Model switches require BOTH --model and compatible --profile.
# Do not export a script until real model prediction and camera tests have passed.
print("CLI status: intentionally pending team model selection and local validation")

## 15. Task 6 evidence for the report and group checklist

Fill using results actually obtained on your laptop. The report needs to explain **whether the model worked**, problems observed, tuning applied, and the reasons behind its real-world performance. Task 6 accounts for **10%** of the assignment; practical limitations are expected and must be documented.

| Field | Record after testing |
|---|---|
| Final `.pkl` and profile | `[TO BE SELECTED BY TEAM]` |
| Reason for selecting the model | `[VALIDATION/TEST RESULTS FROM TEAM]` |
| Laptop OS / Python / environment | `[LOCAL RUN]` |
| Face detector / predictor | `[PROFILE-DEPENDENT]` |
| Input format / normalization | `[MATCH TRAINING]` |
| Camera invocation command | `[FINAL demo.py COMMAND]` |
| Measured FPS / latency | `[LOCAL MEASUREMENTS ONLY]` |
| Expression successes and failure cases | `[MANUAL TRIAL LOG]` |
| Lighting / head rotation / multiple faces | `[LOCAL OBSERVATIONS]` |
| Fine-tuning and effect | `[DOCUMENT ACTUAL CHANGES]` |
| Final limitations / whether adequate | `[EVIDENCE-BASED CONCLUSION]` |

When finished, update `workload.md` T16 and T17 and draft report §8 (Real-time webcam demo). Do not claim final model selection until the team agrees.

**AI usage:** follow the assignment's required AI declaration; disclose use of ChatGPT assistance for notebook scaffolding and code review, and independently verify correctness and measurements.